# AI Code Auditor v2 — Evaluation

Evaluates DeepSeek-Coder-6.7B fine-tuned on Big-Vul v2 (top-10 CWEs).

### Before running:
1. GPU: T4 x1
2. Attach `bigvul-v2-evaluate` dataset (has test.jsonl + lora_adapter_v2)

In [ ]:
!pip install -q transformers==4.40.2 peft==0.10.0 accelerate==0.29.3 bitsandbytes==0.45.3 sacrebleu rouge-score
print('Done')

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
print('Environment set')

In [ ]:
import os, json, re

TOP10 = {'CWE-119','CWE-20','CWE-399','CWE-264','CWE-200',
         'CWE-125','CWE-190','CWE-416','CWE-362','CWE-189'}

# Find paths
TEST_PATH = None
ADAPTER_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        full = os.path.join(root, f)
        if f == 'test.jsonl': TEST_PATH = full
        if f == 'adapter_config.json' and 'checkpoint' not in root:
            ADAPTER_PATH = root

assert TEST_PATH, 'test.jsonl not found'
assert ADAPTER_PATH, 'adapter_config.json not found'
print(f'Test    : {TEST_PATH}')
print(f'Adapter : {ADAPTER_PATH}')

# Load test records — top-10 CWEs only
with open(TEST_PATH) as f:
    all_records = [json.loads(l) for l in f]
test_records = [r for r in all_records if r['cwe'] in TOP10][:100]
print(f'Test samples: {len(test_records)}')
print(f'CWE sample: {test_records[0]["cwe"]} | Text: {len(test_records[0]["text"])} chars')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = 'deepseek-ai/deepseek-coder-6.7b-base'
print(f'CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0)}')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config,
    device_map={'': 0}, trust_remote_code=True, torch_dtype=torch.float16,
)
base_model.config.use_cache = True
print(f'Base model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB')

In [ ]:
# Load fine-tuned adapter
finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
finetuned_model.eval()
print(f'Fine-tuned model ready. Adapter: {ADAPTER_PATH}')

In [ ]:
# DeepSeek prompt format — ends with 'CWE:' to force CWE prediction first
def build_prompt(code):
    return (
        'You are an expert security code auditor.\n'
        'Analyze the following C/C++ code and identify the security vulnerability.\n\n'
        f'```c\n{code}\n```\n\n'
        'Respond with the CWE type first, then explain and provide a secure rewrite.\n'
        'CWE:'
    )

def extract_cwe(text):
    m = re.search(r'CWE-\d+', text)
    return m.group(0) if m else 'Unknown'

def run_inference(model, code, max_new_tokens=200):
    prompt = build_prompt(code)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=400).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Quick sanity check
test_out = run_inference(finetuned_model, 'void foo(char *s) { char buf[8]; strcpy(buf, s); }')
print(f'Sanity check output: {test_out[:100]}')
print(f'Extracted CWE: {extract_cwe(test_out)}')

In [ ]:
# Run baseline inference (zero-shot, no adapter)
from tqdm import tqdm

print('Running BASELINE inference (zero-shot)...')
baseline_results = []
for i, record in enumerate(tqdm(test_records)):
    output = run_inference(base_model, record['vulnerable_code'])
    baseline_results.append({
        'sample_id': i,
        'ground_truth_cwe': record['cwe'],
        'predicted_cwe': extract_cwe(output),
        'ground_truth_secure': record['secure_code'],
        'predicted_secure': output,
        'raw_output': output,
        'vulnerable_code': record['vulnerable_code'],
    })

with open('/kaggle/working/baseline_results_v2.jsonl', 'w') as f:
    for r in baseline_results: f.write(json.dumps(r) + '\n')

bl_acc = sum(1 for r in baseline_results if r['ground_truth_cwe'] == r['predicted_cwe'])
print(f'Baseline CWE Accuracy: {bl_acc}/100 = {bl_acc}%')

In [ ]:
# Run fine-tuned inference
print('Running FINE-TUNED inference...')
finetuned_results = []
for i, record in enumerate(tqdm(test_records)):
    output = run_inference(finetuned_model, record['vulnerable_code'])
    finetuned_results.append({
        'sample_id': i,
        'ground_truth_cwe': record['cwe'],
        'predicted_cwe': extract_cwe(output),
        'ground_truth_secure': record['secure_code'],
        'predicted_secure': output,
        'raw_output': output,
        'vulnerable_code': record['vulnerable_code'],
    })

with open('/kaggle/working/finetuned_results_v2.jsonl', 'w') as f:
    for r in finetuned_results: f.write(json.dumps(r) + '\n')

ft_acc = sum(1 for r in finetuned_results if r['ground_truth_cwe'] == r['predicted_cwe'])
unknown = sum(1 for r in finetuned_results if r['predicted_cwe'] == 'Unknown')
print(f'Fine-tuned CWE Accuracy: {ft_acc}/100 = {ft_acc}%')
print(f'Unknown predictions: {unknown}')
print()
for r in finetuned_results[:3]:
    print(f'GT: {r["ground_truth_cwe"]} | Pred: {r["predicted_cwe"]} | Raw: {r["raw_output"][:80]}')

In [ ]:
# Compute all metrics
import sacrebleu
from rouge_score import rouge_scorer
import numpy as np
from collections import Counter

def compute_metrics(results, name):
    refs  = [r['ground_truth_secure'] for r in results]
    hyps  = [r['predicted_secure']    for r in results]
    gt    = [r['ground_truth_cwe']    for r in results]
    pred  = [r['predicted_cwe']       for r in results]

    bleu   = sacrebleu.corpus_bleu(hyps, [refs]).score
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    rougeL = np.mean([scorer.score(r,h)['rougeL'].fmeasure for r,h in zip(refs,hyps)])
    cwe_acc = sum(1 for g,p in zip(gt,pred) if g==p) / len(gt)

    print(f'\n{"-"*50}')
    print(f'  {name}')
    print(f'{"-"*50}')
    print(f'  BLEU-4            : {bleu:.2f}')
    print(f'  ROUGE-L           : {rougeL:.3f}')
    print(f'  CWE Accuracy      : {cwe_acc:.1%}')
    print(f'  Unknown preds     : {sum(1 for p in pred if p=="Unknown")}')
    print(f'\n  Per-CWE accuracy:')
    for cwe, count in Counter(gt).most_common():
        correct = sum(1 for g,p in zip(gt,pred) if g==cwe and g==p)
        print(f'    {cwe}: {correct}/{count}')

    return {'bleu4': round(bleu,2), 'rougeL': round(rougeL,3), 'cwe_accuracy': round(cwe_acc,3)}

bl_metrics = compute_metrics(baseline_results,  'Baseline (Zero-shot DeepSeek-6.7B)')
ft_metrics = compute_metrics(finetuned_results, 'Fine-tuned (QLoRA DeepSeek-6.7B, top-10 CWEs)')

import json
with open('/kaggle/working/evaluation_metrics_v2.json', 'w') as f:
    json.dump({'baseline': bl_metrics, 'finetuned': ft_metrics}, f, indent=2)
print('\nSaved evaluation_metrics_v2.json')

In [ ]:
# Download links
from IPython.display import FileLink, display
display(FileLink('evaluation_metrics_v2.json'))
display(FileLink('finetuned_results_v2.jsonl'))
display(FileLink('baseline_results_v2.jsonl'))